# context_beready.ipynb

Original Bi-CoT research notebook, recovered from the author’s AI_Study archive. Outputs, execution state, and machine metadata were removed; API credentials now come from `OPENAI_API_KEY`, and local paths now use `../data/workspace` relative to this notebook directory. See `docs/WORKFLOW.md` before selecting cells. Cells include experimental alternatives and data writes; this is not a single Run All pipeline.


In [ ]:
import json
from typing import List, Dict, Set

def extract_unique_title_sents_from_jsons(q_json_paths: List[str]) -> List[Dict[str, str]]:
    """
    여러 q.json 파일 경로를 받아, 모든 candidate_chains 내에서
    중복되지 않는 {"title": title, "sents": paragraph} 리스트를 반환
    """
    seen_paragraphs: Set[str] = set()
    result: List[Dict[str, str]] = []

    for q_json_path in q_json_paths:
        with open(q_json_path, "r", encoding="utf-8") as f:
            q_data = json.load(f)

        candidate_chains = q_data.get("candidate_chains", [])

        for chain in candidate_chains:
            for doc in chain:
                title = doc.get("title", "")
                sents = doc.get("sents", [])
                paragraph = " ".join(sents).strip()

                if paragraph and paragraph not in seen_paragraphs:
                    result.append({"title": title, "sents": paragraph})
                    seen_paragraphs.add(paragraph)

    return result

In [ ]:
# 예시: 여러 q.json 경로 리스트
q_json_paths = [
    "../data/workspace/testing/5a8b57f25542995d1e6f1371/2/2/Q.json",
    "../data/workspace/testing/5a8b57f25542995d1e6f1371/2/3/Q.json",
    "../data/workspace/testing/5a8b57f25542995d1e6f1371/2/1/Q.json",
]

title_sents_list = extract_unique_title_sents_from_jsons(q_json_paths)

for item in title_sents_list:
    print(item["title"], "|", item["sents"])

In [ ]:
from openai import OpenAI
import os
import re
import json
from typing import List, Dict, Tuple, Set

client = OpenAI(api_key=__import__("os").environ["OPENAI_API_KEY"])

def call_openai(prompt, model="gpt-3.5-turbo", max_tokens=256, temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()

def is_answer_supported_by_context(question: str, answer: str, context: str) -> Tuple[bool, str]:
    """
    실제 reasoning+evidence 프롬프트를 GPT에게 보내고 답변을 파싱.
    """
    prompt = (
        f"Is that true \"{answer}\" is the answer of \"{question}\"?\n"
        "If yes, quote the evidence sentence(s) from the context.\n"
        "If no, say 'No' and explain why.\n\n"
        f"Context:\n{context}"
    )
    out = call_openai(prompt)
    # 간단 패턴: yes+evidence or no+이유
    if re.match(r"(?i).*yes.*", out):
        # evidence만 추출
        m = re.search(r"(?:[\"']?evidence[\"']?[:\-]?\s*)(.*)", out, re.IGNORECASE)
        evidence = m.group(1).strip() if m else out
        return True, evidence
    else:
        return False, out.strip()

def merge_context_details(context_details_list: List[List[Dict[str, str]]]) -> List[Dict[str, str]]:
    seen_sents: Set[str] = set()
    result: List[Dict[str, str]] = []
    for context_details in context_details_list:
        for item in context_details:
            if item["sents"] not in seen_sents:
                result.append(item)
                seen_sents.add(item["sents"])
    return result

def save_cot2_a_json(save_path: str, question: str, answer: str, evidence: str):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    answer_obj = {
        "question": question,
        "answer": answer,
        "evidence": evidence
    }
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(answer_obj, f, ensure_ascii=False, indent=2)

def save_cot2_q_json(save_path: str, question: str):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    question_obj = {
        "question": question
    }
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(question_obj, f, ensure_ascii=False, indent=2)

def reformulate_question(original_question: str, context_details_list: List[List[Dict[str, str]]]) -> str:
    """
    기존 context의 문서(title)와 sents, 오리지널 질문을 바탕으로,  
    검색 방향성과 필요한 정보가 강조된 새로운 질문 생성. (API 사용)
    """
    titles = set()
    for context_details in context_details_list:
        for item in context_details:
            titles.add(item["title"])
    joined_titles = ", ".join(list(titles))
    context_summary = "\n".join([f"[{item['title']}] {item['sents']}" for context_details in context_details_list for item in context_details])
    prompt = (
        f"Original Question:\n{original_question}\n\n"
        f"Already retrieved documents:\n{joined_titles}\n\n"
        f"Retrieved context sentences:\n{context_summary}\n\n"
        f"Reformulate a better search question that avoids the above retrieved content, "
        "uses more specific and relevant keywords, and guides the retriever to new directions."
    )
    return call_openai(prompt)

def process_subq_cot2(example_id_dir, subq_num):
    subq_path = os.path.join(example_id_dir, str(subq_num))
    cot_1_a_path = os.path.join(subq_path, "cot_1", "a.json")
    if not os.path.exists(cot_1_a_path):
        print(f"❌ cot_1/a.json 없음: {cot_1_a_path}")
        return
    with open(cot_1_a_path, "r", encoding="utf-8") as f:
        cot_1_a = json.load(f)
    question = cot_1_a["question"]
    answer = cot_1_a["answer"]
    context = cot_1_a["context"]

    # 1차: cot_1/a.json의 context에서 answer 지원 확인
    is_yes, evidence = is_answer_supported_by_context(question, answer, context)
    cot_2_dir = os.path.join(subq_path, "cot_2")
    a_path = os.path.join(cot_2_dir, "a.json")
    q_path = os.path.join(cot_2_dir, "q.json")

    if is_yes and evidence:
        save_cot2_a_json(a_path, question, answer, evidence)
        print(f"[{subq_num}] evidence found in cot_1 context.")
        return

    # 2차: aug 1,2와 0(원질문) 폴더의 context_detail 모두 합치기
    context_details_list = []
    for aug in ['1', '2']:
        a_path2 = os.path.join(subq_path, aug, "a.json")
        if os.path.exists(a_path2):
            with open(a_path2, "r", encoding="utf-8") as f:
                data = json.load(f)
            if "context_detail" in data:
                context_details_list.append(data["context_detail"])
    zero_a_path = os.path.join(example_id_dir, "0", "a.json")
    if os.path.exists(zero_a_path):
        with open(zero_a_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if "context_detail" in data:
            context_details_list.append(data["context_detail"])
    total_context_detail = merge_context_details(context_details_list)
    total_context = "\n".join([item["sents"] for item in total_context_detail])

    # 2차 검증: 합친 context에서 answer 증명 시도
    is_yes2, evidence2 = is_answer_supported_by_context(question, answer, total_context)
    if is_yes2 and evidence2:
        save_cot2_a_json(a_path, question, answer, evidence2)
        print(f"[{subq_num}] evidence found in aug+0 context.")
        return

    # 3차: 질문 재구성
    new_question = reformulate_question(question, context_details_list)
    save_cot2_q_json(q_path, new_question)
    print(f"[{subq_num}] 질문 재구성 완료, cot_2/q.json에 저장됨.")

def run_cot2(example_id_dir):
    subq_dirs = [d for d in os.listdir(example_id_dir)
                 if os.path.isdir(os.path.join(example_id_dir, d)) and d.isdigit() and d != "0"]
    subq_dirs = sorted([int(d) for d in subq_dirs])
    for subq_num in subq_dirs:
        process_subq_cot2(example_id_dir, subq_num)

# 사용 예시
example_id_dir = "../data/workspace/testing/5a8b57f25542995d1e6f1371"
run_cot2(example_id_dir)